# 01 — Separar stems com Demucs

Este notebook usa o **Demucs** (Meta, open-source) para separar sua gravação de  
voz + violão em pistas individuais: `vocals.wav` e `no_vocals.wav` (ou 4–6 stems).

**Entrada:** arquivo em `audio/raw/` (WAV, MP3, MP4, M4A, FLAC, OGG ou AAC).  
Formatos não-WAV são convertidos automaticamente para WAV antes do Demucs.  
**Saída:** `audio/stems/<modelo>/<nome>/vocals.wav` e `no_vocals.wav`

---
**Modelos disponíveis:**
| Modelo | Stems | Qualidade | Velocidade |
|---|---|---|---|
| `htdemucs` | 4 (vocals, drums, bass, other) | boa | rápido |
| `htdemucs_6s` | 6 (+ guitar, piano) | melhor | médio |
| `mdx_extra` | 4 | alta | lento |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from scripts.utils import (
    listar_audio, carregar_audio, plotar_audio, log_sessao,
    garantir_wav,
    AUDIO_RAW, AUDIO_STEMS,
)

print('Arquivos disponíveis em audio/raw:')
arquivos = listar_audio(AUDIO_RAW)

In [ ]:
# ── CONFIGURAÇÃO — edite aqui ─────────────────────────────────────────────────

ARQUIVO_ENTRADA = 'minha_musica.wav'   # nome do arquivo em audio/raw/ (wav/mp3/mp4/m4a/flac/ogg/aac)
MODELO_DEMUCS   = 'htdemucs'          # htdemucs | htdemucs_6s | mdx_extra
DOIS_STEMS      = True                # True = só vocals/no_vocals | False = todos os stems

# ─────────────────────────────────────────────────────────────────────────────

caminho_entrada = AUDIO_RAW / ARQUIVO_ENTRADA
assert caminho_entrada.exists(), f'Arquivo não encontrado: {caminho_entrada}'

# Demucs e torchaudio precisam de WAV — converte se for outro formato (idempotente)
caminho_entrada = garantir_wav(caminho_entrada)

audio, sr = carregar_audio(caminho_entrada)
plotar_audio(audio, sr, titulo=f'Entrada: {caminho_entrada.name}')

In [ ]:
import subprocess

cmd = [
    'python', '-m', 'demucs',
    '-n', MODELO_DEMUCS,
    '--out', str(AUDIO_STEMS),
    str(caminho_entrada),
]

if DOIS_STEMS:
    cmd += ['--two-stems', 'vocals']

print('Executando:', ' '.join(cmd))
print('...')
resultado = subprocess.run(cmd, capture_output=False)

if resultado.returncode == 0:
    print('✓ Separação concluída')
else:
    print('✗ Erro na separação — verifique a mensagem acima')

In [ ]:
# Verificar e visualizar os stems gerados
pasta_stems = AUDIO_STEMS / MODELO_DEMUCS / caminho_entrada.stem
print(f'Stems em: {pasta_stems}\n')

for stem_path in sorted(pasta_stems.glob('*.wav')):
    audio_stem, sr_stem = carregar_audio(stem_path)
    plotar_audio(audio_stem, sr_stem, titulo=stem_path.name)

# Caminho da voz isolada (para usar no próximo notebook)
vocal_path = pasta_stems / 'vocals.wav'
print(f'\nVoz isolada para próxima etapa:\n  {vocal_path}')

In [ ]:
# ── REGISTRO DA SESSÃO ────────────────────────────────────────────────────────
# Edite as notas abaixo antes de executar

log_sessao(
    titulo=f'Separação de stems — {ARQUIVO_ENTRADA}',
    notas=f"""
- Arquivo original: {ARQUIVO_ENTRADA}
- Modelo Demucs: {MODELO_DEMUCS}
- Modo: {'2 stems (vocals/no_vocals)' if DOIS_STEMS else 'stems completos'}
- Resultado: OK
- Observações: (escreva aqui suas impressões sobre a qualidade da separação)
"""
)

## Próximo passo

Com a voz isolada em `audio/stems/htdemucs/<nome>/vocals.wav`,  
abra o notebook `02_vocal2bgm.ipynb` para gerar o arranjo musical.